In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from notebooks.utils.semeval_random_forest_token_visualization_utils import (
    build_gold_boundaries,
    load_selected_annotations,
    load_selected_token_labels,
    load_selected_windows,
    load_visualization_models,
    resolve_prepared_inputs,
    scan_annotation_ids,
    scan_stage1_ids,
    score_visualization_windows,
    select_eligible_text_ids,
    write_visualization_export,
)
from notebooks.utils.semeval_text_comparison_utils import (
    plot_reconstructed_token_scores,
    plot_window_level_scores,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
SCORER_LLM = "gemma_3_270m_it"
MODEL_FAMILY = "random_forest"
FEATURE_SET = "simple_similarity"
TRAINING_REGIME = "Mixed"
ARTIFACT_RUN_ID = "feature-vector-classification-2026-09-21-logreg-random-forest"
WINDOW_LENGTHS = (32, 64, 128)
SELECTION_SEED = 42
TEXT_COUNT = 20
DATASET = "semeval2024_task8_subtask_c_train"
SPLIT = "MG"
ANNOTATION_PATH = DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC/SemEval2024_task8_C_train_MG.csv"
OUTPUT_DIRECTORY = (
    DATA_ROOT / "experiment-results" / "semeval-random-forest-token-visualization"
    / f"{SCORER_LLM}__{MODEL_FAMILY}__{FEATURE_SET}__{TRAINING_REGIME}__{ARTIFACT_RUN_ID}__seed{SELECTION_SEED}__count{TEXT_COUNT}"
)


In [ ]:
models = load_visualization_models(DATA_ROOT, run_id=ARTIFACT_RUN_ID)
prepared = resolve_prepared_inputs(DATA_ROOT, DATASET, models, split=SPLIT)
selected = select_eligible_text_ids(
    {window: path for window, (path, _) in prepared.items()},
    stage1_ids=scan_stage1_ids(DATA_ROOT / "processed", DATASET, SCORER_LLM, SPLIT),
    annotation_ids=scan_annotation_ids(ANNOTATION_PATH),
    count=TEXT_COUNT,
    seed=SELECTION_SEED,
)
selected_ids = set(selected["text_id"].astype(int))
windows = load_selected_windows(prepared, selected_ids)
scored_windows = score_visualization_windows(models, windows)
annotations = load_selected_annotations(ANNOTATION_PATH, selected_ids)
token_labels = load_selected_token_labels(
    DATA_ROOT / "processed", DATASET, SCORER_LLM, SPLIT, selected_ids
)
gold_boundaries = build_gold_boundaries(annotations, token_labels)
metadata = {
    "artifact_run_id": ARTIFACT_RUN_ID,
    "scorer_llm": SCORER_LLM,
    "model_family": MODEL_FAMILY,
    "feature_set": FEATURE_SET,
    "training_regime": TRAINING_REGIME,
    "window_lengths": list(WINDOW_LENGTHS),
    "stride": 16,
    "dataset": DATASET,
    "split": SPLIT,
    "selection_seed": SELECTION_SEED,
    "text_count": TEXT_COUNT,
    "selected_text_ids": selected["text_id"].astype(int).tolist(),
}
output = write_visualization_export(
    OUTPUT_DIRECTORY,
    selected_texts=selected,
    scored_windows=scored_windows,
    gold_boundaries=gold_boundaries,
    metadata=metadata,
)


In [ ]:
figures = output / "figures"
figures.mkdir()
selection_group = f"random sample, seed={SELECTION_SEED}"

for text_id in selected["text_id"].astype(int):
    boundary = gold_boundaries.loc[gold_boundaries["text_id"].eq(text_id)].iloc[0]
    gold_boundary = int(boundary["gold_token_boundary"])
    token_count = int(boundary["token_count"])
    window_figure = plot_window_level_scores(
        scored_windows,
        text_id=text_id,
        gold_boundary=gold_boundary,
        selection_group=selection_group,
    )
    token_figure = plot_reconstructed_token_scores(
        scored_windows,
        text_id=text_id,
        token_count=token_count,
        gold_boundary=gold_boundary,
        selection_group=selection_group,
    )
    window_figure.savefig(figures / f"text-{text_id:06d}-window-scores.png", dpi=160)
    token_figure.savefig(figures / f"text-{text_id:06d}-token-scores.png", dpi=160)
    display(window_figure)
    display(token_figure)
    plt.close(window_figure)
    plt.close(token_figure)
